In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../../maxtext")))
os.environ["SKIP_JAX_PRECOMPILE"] = "1"

import functools
import transformers
from transformers import LlamaForCausalLM, LlamaTokenizer
import torch

import jax
import jax.numpy as jnp
from flax import nnx
import flax.linen as nn

import MaxText as mt
from MaxText import pyconfig
from MaxText.integration.tunix.tunix_adaptor import TunixMaxTextLlama

from tunix.rl.rollout.vllm_rollout import VllmRollout
from tunix.rl.rollout import base_rollout
from tunix.models.llama3 import model as llama3_lib

from vllm import LLM

/home/wenxindong_google_com/miniconda3/envs/tunix/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


2025-08-14 20:27:32.323442: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755203252.336381  621875 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755203252.340169  621875 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755203252.356976  621875 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755203252.356992  621875 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755203252.356994  621875 computation_placer.cc:177] computation placer alr

INFO 08-14 20:27:37 [__init__.py:241] Automatically detected platform tpu.
WARNING 08-14 20:27:37 [__init__.py:14] 🚨  CAUTION: You are using 'tpu_commons' , which is experimental and NOT intended for production use yet. Please see the README for more details.
INFO 08-14 20:27:37 [__init__.py:16] TPU info: node_name=wenxindong-v6e-8 | tpu_type=v6e-8 | worker_id=0 | num_chips=8 | num_cores_per_chip=1
INFO 08-14 20:27:37 [__init__.py:29] Running vLLM without Pathways. Module pathwaysutils is not imported.


In [ ]:
MODEL = "meta-llama/Llama-3.1-8B"
TOTAL_TPU_TO_USE = 8
MESH = [(1, TOTAL_TPU_TO_USE), ("fsdp", "tp")]  # YY


model_tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL)
mesh = model_mesh = jax.make_mesh(
    *MESH, devices=jax.devices()[:TOTAL_TPU_TO_USE]
)

In [ ]:
# vLLM model 
golden_llm = LLM(
    MODEL,
    max_model_len=1024,
    tensor_parallel_size=8,
    gpu_memory_utilization=0.3,
)

In [ ]:
def get_ref_maxtext_model(config):

  # python3 -m MaxText.train MaxText/configs/base.yml base_output_directory=${BASE_OUTPUT_DIRECTORY} dataset_path=${DATASET_PATH} tokenizer_path=assets/tokenizer.gemma load_parameters_path=${CONVERTED_CHECKPOINT} per_device_batch_size=1 run_name=${FINETUNE_RUN_NAME} max_target_length=8192 steps=10 async_checkpointing=false model_name=gemma-2b checkpoint_period=5

  def create_model(config):
    return mt.from_pretrained(config, rngs=nnx.Rngs(params=0, dropout=1))

  abstract_model = nnx.eval_shape(create_model, config=config)
  graphdef, abstract_state = nnx.split(abstract_model)
  print("The abstract NNX state (all leaves are abstract arrays):")
  nnx.display(abstract_state)
  specs = nnx.get_partition_spec(abstract_state)
  mesh = abstract_model.mesh

  # JIT a function that creates the model state with proper sharding from the start.
  # By providing out_shardings, we instruct JAX to produce sharded output directly,
  # avoiding a large intermediate allocation on a single device.
  with nn.logical_axis_rules(config.logical_axis_rules):
    out_shardings = nn.logical_to_mesh_sharding(specs, mesh)

  @functools.partial(jax.jit, out_shardings=out_shardings)
  def create_sharded_state():
    # This will be JIT-compiled. JAX knows the output sharding and can
    # initialize the parameters directly on the target devices in a sharded way.
    model = create_model(config)
    return nnx.state(model)

  with mesh:

    # Create the model with sharded parameters.
    sharded_state = create_sharded_state()
    model = nnx.merge(graphdef, sharded_state)

    target_for_restore = jax.tree.map(
        lambda v: v.value,
        sharded_state,
        is_leaf=lambda n: isinstance(n, nnx.Variable),
    )
    checkpoint = mt.checkpointing.load_params_from_path(
        load_parameters_from_path=config.load_parameters_path,
        abstract_unboxed_params=target_for_restore,
        checkpoint_storage_concurrent_gb=None,
    )
    # checkpoint = ocp.StandardCheckpointer().restore(
    #     "gs://maxtext-gemma/2b/2025-08-05-04-37/0/items", target=target_for_restore
    # )
    if checkpoint:
      nnx.update(model, checkpoint)

    tunix_model = TunixMaxTextLlama(
        base_model=model,
        use_attention_mask=False,  # trust Tunix loss masking
    )

    model_config = llama3_lib.ModelConfig.llama3_1_8b()
    tunix_model.config = model_config

  return tunix_model, mesh, model_config


config_ref = pyconfig.initialize(
    [
        "",
        "../../maxtext/MaxText/configs/base.yml",
    ],  # TODO: @mazumdera: why decode.py?
    base_output_directory="gs://dummy_output_dir",  # This is not used in Tunix.
    run_name="test-tunix-maxtext-llama3.1-8b",
    # run_name="test-tunix-maxtext-llama3.1-8b",
    # dataset_path=we use Tunix's dataset
    # TODO: @mazumdera: change this to use checkpoint
    tokenizer_type="tiktoken",
    tokenizer_path="assets/tokenizer_llama3.tiktoken",
    load_parameters_path="gs://maxtext-model-checkpoints/llama3.1-8b/2025-01-23-19-04/scanned/0/items",
    # tokenizer_path="assets/tokenizer.gemma",
    per_device_batch_size=1,
    max_prefill_predict_length=4,
    max_target_length=16,
    steps=10,
    async_checkpointing="false",
    model_name="llama3.1-8b",
    # model_name="gemma-2b",
    checkpoint_period=5,
    skip_jax_distributed_system="true",
    weight_dtype="bfloat16",
    attention="dot_product",
    remat_policy="custom",
    decoder_layer_input="offload",
    query_proj="offload",
    key_proj="offload",
    value_proj="offload",
    opt_type="sgd",
)

# MaxText model
maxtext_model, mesh, model_config = get_ref_maxtext_model(config_ref)
nnx.display(maxtext_model)


In [ ]:

hf_model = LlamaForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
)

In [ ]:
# Compare weights 

_golden_state_flatten = (
    golden_llm.llm_engine.model_executor.driver_worker.model_runner.state.flat_state()
)
_maxtext_state_flatten = nnx.state(maxtext_model).flat_state()

golden_state_flatten = {
    '.'.join(str(key) for key in keys): v for keys, v in _golden_state_flatten
}
maxtext_state_flatten = {
    '.'.join(str(key) for key in keys): v for keys, v in _maxtext_state_flatten
}

In [ ]:
# MaxText
maxtext_state_flatten["base.token_embedder.embedding"].value

In [ ]:
# vLLM
golden_state_flatten["embed.embedding"].value


In [ ]:
# HF
hf_model.model.embed_tokens.weight

In [ ]:
##### Code after this section is not used. 

In [11]:
def create_maxtext_to_vllm_mappings():
  """Create mappings for transferring MaxText scanned state to vLLM unscanned state."""
  return {
      # Token embeddings - shard vocab dimension for TP
      'base.token_embedder.embedding': ('embed.embedding', ('model', None)),
      # Final layer norm - no sharding needed
      'base.decoder.decoder_norm.scale': ('model.norm.scale', (None,)),
      # LM head (logits projection) - shard vocab dimension for TP
      'base.decoder.logits_dense.kernel': ('lm_head', (None, 'model')),
      # Layer-specific mappings (scanned -> unscanned)
      # MLP components - shard hidden dimensions for TP
      'base.decoder.layers.mlp.wi_0.kernel': (
          'model.layers.*.mlp.gate_proj.kernel',
          (None, 'layer', 'model'),
      ),  # gate_proj: (4096, 14336) - shard output
      'base.decoder.layers.mlp.wi_1.kernel': (
          'model.layers.*.mlp.up_proj.kernel',
          (None, 'layer', 'model'),
      ),  # up_proj: (4096, 14336) - shard output
      'base.decoder.layers.mlp.wo.kernel': (
          'model.layers.*.mlp.down_proj.kernel',
          ('model', 'layer', None),
      ),  # down_proj: (14336, 4096) - shard input
      # Layer norms - no sharding needed
      'base.decoder.layers.pre_self_attention_layer_norm.scale': (
          'model.layers.*.input_layernorm.scale',
          (None, 'layer'),
      ),
      'base.decoder.layers.post_self_attention_layer_norm.scale': (
          'model.layers.*.post_attention_layernorm.scale',
          (None, 'layer'),
      ),
      # Attention components - shard head dimensions for TP
      'base.decoder.layers.self_attention.query.kernel': (
          'model.layers.*.self_attn.q_proj.kernel',
          (None, 'layer', 'model', None),
      ),  # q_proj: shard num_heads
      'base.decoder.layers.self_attention.key.kernel': (
          'model.layers.*.self_attn.k_proj.kernel',
          (None, 'layer', 'model', None),
      ),  # k_proj: shard num_kv_heads
      'base.decoder.layers.self_attention.value.kernel': (
          'model.layers.*.self_attn.v_proj.kernel',
          (None, 'layer', 'model', None),
      ),  # v_proj: shard num_kv_heads
      'base.decoder.layers.self_attention.out.kernel': (
          'model.layers.*.self_attn.o_proj.kernel',
          ('model', 'layer', None, None),
      ),  # o_proj: shard input heads
  }

In [12]:
transpose_keys = {
    # MLP transposes (after layer extraction)
    'wo.kernel': (1, 0),  # down_proj: (14336, 4096) - transpose needed
    # Attention output transpose (after layer extraction)
    'out.kernel': (
        1,
        2,
        0,
    ),  # o_proj: (32, 128, 4096) -> (32, 128, 4096) - reorder dimensions
}


maxtext_model.to_hf_mappings = create_maxtext_to_vllm_mappings()
maxtext_model.to_hf_transpose_keys = lambda *args: transpose_keys
maxtext_model.lora_to_hf_mappings = lambda *args: None  # No LoRA

In [ ]:
import functools
import logging
import re
from typing import Any, Dict, Iterator, List, Optional

from flax import nnx
import jax
from jax import lax
import jax.numpy as jnp


def build_flat_dict(
    flat_state: Iterator[tuple[tuple[str, ...], nnx.State]],
    mappings: Dict[str, tuple[str, tuple[int, ...]]],
):
  """Build a new flat dictionary from the flat state using the provided mappings.

  Args:
    flat_state: A list of tuples, where each tuple contains the nested keys and
      the corresponding value.
    mappings: A dictionary defining how to map keys from the source state to the
      target state. The keys of the dictionary are the source keys, and the
      values are tuples containing the target key and the sharding information.

  Returns:
    A new flat dictionary with the mapped keys and values.
  """
  new_flat_dict = {}
  for keys, v in flat_state:
    path = '.'.join(str(key) for key in keys)
    mapped = False
    for src, (tgt, sharding) in mappings.items():
      regex = '^' + re.escape(tgt).replace('\\.\\*', r'\.(\d+)') + '$'
      matched = re.match(regex, path)
      if matched:
        # Extract wildcards if any
        wildcards = matched.groups()
        src_parts = []
        wc_index = 0
        for part in src.split('.'):
          if part == '*':
            src_parts.append(wildcards[wc_index])
            wc_index += 1
          else:
            src_parts.append(part)
        actual_src = '.'.join(src_parts)
        # Check if this is a scanned parameter (has 'layer' in sharding spec)
        if sharding and 'layer' in sharding:
          if actual_src not in new_flat_dict:
            new_flat_dict[actual_src] = ([], sharding)
          layer_number = int(matched.groups()[0])
          new_flat_dict[actual_src][0].append((layer_number, v))
        else:
          # Regular (non-scanned) parameter
          new_flat_dict[actual_src] = v, sharding

        mapped = True
        break
    # There are no mappings for rng related params.
    if not mapped:
      logging.warning('!!! No mapping for flat state: %s', path)

  # Sort layers
  for key, (layers, sharding) in new_flat_dict.items():
    if isinstance(layers, list):
      layers.sort(key=lambda x: x[0])
      new_flat_dict[key] = ([layer for _, layer in layers], sharding)

  return new_flat_dict


def transfer_state_with_mappings(
    src_state,
    dst_state,
    key_mappings,
    transpose_keys=None,
    reshard_fn=None,
):
  """Transfer state using mappings, with optional transpose and shard logic.

  Args:
    src_state: The source state to transfer from.
    dst_state: The destination state to transfer to.
    key_mappings: A dictionary defining how to map keys from the source state to
      the target state. The keys of the dictionary are the source keys, and the
      values are tuples containing the target key and the sharding information.
    transpose_keys: A dictionary defining which keys to transpose and the
      corresponding axes to transpose.
    reshard_fn: A function to shard the value.

  Returns:
    The target state with the transferred values.
  """

  src_flat = src_state.flat_state()
  tgt_flat = dst_state.flat_state()
  # Maps source keys to target tensor(s) and sharding spec
  new_src_dict = build_flat_dict(tgt_flat, key_mappings)

  def _process_and_assign_value(value, tgt_param, flat_key, src_keys):
    # Optional transpose
    if (
        transpose_keys
        and (src_keys[-1] in transpose_keys)
        and 'lora' not in src_keys[-1]
    ):
      value = jnp.transpose(value, transpose_keys[src_keys[-1]])

    # Shape check and general padding support
    if tgt_param.value.shape != value.shape:
      if len(value.shape) != len(tgt_param.value.shape):
        raise ValueError(
            f'Rank mismatch for {flat_key}: {value.shape} vs'
            f' {tgt_param.value.shape}'
        )

      pad_width = []
      for i, (src_dim, tgt_dim) in enumerate(
          zip(value.shape, tgt_param.value.shape)
      ):
        if src_dim < tgt_dim:
          # Optional: enforce vLLM padding constraint only on padded dims
          assert tgt_dim == 128, (
              f'vLLM only supports padding to 128, but got {tgt_dim} in dim'
              f' {i} for {flat_key}'
          )
          pad_width.append((0, tgt_dim - src_dim))
        elif src_dim == tgt_dim:
          pad_width.append((0, 0))
        else:
          raise ValueError(
              f'Cannot shrink shape for {flat_key}: {value.shape} ->'
              f' {tgt_param.value.shape}'
          )

      logging.info(
          'Padding %s from shape %s to %s',
          flat_key,
          value.shape,
          tgt_param.value.shape,
      )
      value = jnp.pad(value, pad_width)

    # Type cast if needed
    if tgt_param.value.dtype != value.dtype:
      logging.warning(
          'Type mismatch on %s: %s -> %s',
          flat_key,
          value.dtype,
          tgt_param.value.dtype,
      )
      value = value.astype(tgt_param.value.dtype)

    # Apply resharding if provided
    new_value = (
        reshard_fn(value, tgt_param.value.sharding) if reshard_fn else value
    )
    tgt_param.value = new_value

  def _extract_layer_from_scanned_tensor(tensor, layer_idx, layer_axis):
    """Extract a specific layer from a scanned tensor."""
    idx = [slice(None)] * tensor.ndim
    idx[layer_axis] = layer_idx
    return tensor[tuple(idx)]

  def _get_layer_axis_from_sharding_spec(sharding_spec):
    """Determine which axis contains the layer dimension from sharding specification."""
    if isinstance(sharding_spec, (list, tuple)):
      for i, spec in enumerate(sharding_spec):
        if spec == 'layer':
          return i
    return None

  def _should_skip_parameter(param_key):
    """Check if a parameter should be skipped during transfer."""
    skip_patterns = [
        'rng',
    ]

    return any(pattern in param_key for pattern in skip_patterns)

  def process_entry(src_keys, src_val):
    flat_key = '.'.join(str(k) for k in src_keys)

    if flat_key not in new_src_dict:
      # Skip RNG states that don't need mapping
      if _should_skip_parameter(flat_key):
        logging.debug('Skipping parameter: %s', flat_key)
      else:
        logging.error('!!! No mapping for source key: %s', flat_key)
        return
      return

    tgt_param, sharding_spec = new_src_dict[flat_key]
    value = src_val.value

    layer_axis = _get_layer_axis_from_sharding_spec(sharding_spec)

    if layer_axis is not None:
      # This is a scanned parameter
      num_layers = len(tgt_param)
      for layer_idx in range(0, num_layers):
        layer_tensor = _extract_layer_from_scanned_tensor(
            value, layer_idx, layer_axis
        )
        _process_and_assign_value(
            layer_tensor, tgt_param[layer_idx], flat_key, src_keys
        )
    else:
      # This is a normal parameter
      _process_and_assign_value(value, tgt_param, flat_key, src_keys)

  # Loop through each parameter
  for src_keys, src_val in src_flat:
    process_entry(src_keys, src_val)

  return dst_state.from_flat_path(tgt_flat)

In [ ]:
import jax
import numpy as np
from etils import epath
import orbax.checkpoint as ocp
import tensorstore as ts
import collections
import operator
import asyncio

ParamInfo = ocp.type_handlers.ParamInfo
ts_context = ts.Context({
    'file_io_concurrency': {'limit': 128},
    'cache_pool#ocdbt': {'total_bytes_limit': 100000000},
})

path = 'gs://maxtext-model-checkpoints/llama3.1-8b/2025-01-23-19-04/scanned/0/items'
param_name = 'base.token_embedder.embedding'
info = ParamInfo(
    name=param_name,
    path=path / param_name,
    parent_dir=path,
    is_ocdbt_checkpoint=True,
    use_zarr3=False,
)
tspec = ocp.type_handlers.get_json_tspec_read(info, use_ocdbt=True)

t = ts.open(ts.Spec(tspec), open=True, context=ts_context).result()
arr = t.read().result()
print(arr)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
import orbax

orbax_checkpointer = orbax.checkpoint.PyTreeCheckpointer()

raw_restored = orbax_checkpointer.restore(
    'gs://maxtext-model-checkpoints/llama3.1-8b/2025-01-23-19-04/scanned/0/items'
)

/home/wenxindong_google_com/miniconda3/envs/tunix/lib/python3.12/site-packages/orbax/checkpoint/_src/serialization/type_handlers.py:1269: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(
I0814 20:59:12.627767  631903 google_auth_provider.cc:181] Running on GCE, using service account 903354779218-compute@developer.gserviceaccount.com
ERROR:root:The available devices are different from the devices used to save the checkpoint.  Please restore checkpoint by passing new shardings for target devices. Original=[DeviceMetadata(id=0), DeviceMetadata(id=1), DeviceMetadata(id=2), DeviceMetadata(id=3), DeviceMetadata(id=4), DeviceMetadata(id=5), DeviceMetadata(id=6), DeviceMetadata(id=7), DeviceMetadata(id=8), DeviceMetadata(id=9), DeviceM

ValueError: sharding passed to deserialization should be specified, concrete and an instance of `jax.sharding.Sharding`. Got None

In [ ]:
for key, v in maxtext_state_flatten.items():
  print(key, v.shape)
print()
for key, v in golden_state_flatten.items():
  print(key, v.shape)

base.decoder.decoder_norm.scale (4096,)
base.decoder.layers.mlp.wi_0.kernel (4096, 32, 14336)
base.decoder.layers.mlp.wi_1.kernel (4096, 32, 14336)
base.decoder.layers.mlp.wo.kernel (14336, 32, 4096)
base.decoder.layers.post_self_attention_layer_norm.scale (4096, 32)
base.decoder.layers.pre_self_attention_layer_norm.scale (4096, 32)
base.decoder.layers.self_attention.key.kernel (4096, 32, 8, 128)
base.decoder.layers.self_attention.out.kernel (32, 32, 128, 4096)
base.decoder.layers.self_attention.query.kernel (4096, 32, 32, 128)
base.decoder.layers.self_attention.value.kernel (4096, 32, 8, 128)
base.decoder.logits_dense.kernel (4096, 128256)
base.decoder.to_nnx__rngs.dropout.count ()
base.decoder.to_nnx__rngs.dropout.key ()
base.decoder.to_nnx__rngs.params.count ()
base.decoder.to_nnx__rngs.params.key ()
base.token_embedder.embedding (128256, 4096)

embed.embedding (128256, 4096)
lm_head (4096, 128256)
model.layers.0.input_layernorm.scale (4096,)
model.layers.0.mlp.down_proj.kernel (143

In [ ]:
from tunix.generate.vllm_sampler import VllmSampler, MappingConfig

sampler = VllmSampler(
    mesh=mesh,
    tokenizer=model_tokenizer,
    max_model_len=1024,
    model_version="meta-llama/Llama-3.1-8b",
    mapping_config=MappingConfig({}, {}, {}, None),
    hbm_utilization=0.3,
)

INFO 08-13 23:23:54 [utils.py:326] non-default args: {'model': 'meta-llama/Llama-3.1-8b', 'max_model_len': 1024, 'tensor_parallel_size': 8, 'gpu_memory_utilization': 0.3, 'disable_log_stats': True}
WARNING 08-13 23:23:54 [__init__.py:516] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 08-13 23:23:56 [__init__.py:707] Resolved architecture: LlamaForCausalLM
INFO 08-13 23:23:56 [__init__.py:1731] Using max model len 1024
INFO 08-13 23:23:56 [__init__.py:2031] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 08-13 23:23:56 [tpu_jax.py:147] The model dtype is not properly set for JAX backend. Overwriting it to jnp.bfloat16
INFO 08-13 23:23:56 [tpu_jax.py:182] Force using UniProcExecutor for JAX on single host.
INFO 08-13 23:23:59 [core.py:72] Initializing a V1 LLM engine (v0.1.dev8444+g094bed7da.d20250811) with config: model='meta-llama/Llama-3.1-8b

In [ ]:
TOTAL_GENERATION_STEPS = 64
MAX_PROMPT_LENGTH = 64
TEMPERATURE = 0.9
TOP_P = 1.0
TOP_K = None
cache_config = base_rollout.RolloutConfig(
    max_tokens_to_generate=TOTAL_GENERATION_STEPS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    kv_cache_size=MAX_PROMPT_LENGTH + TOTAL_GENERATION_STEPS + 256,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
)

In [ ]:
# show_hbm_usage = utils.show_hbm_usage

# show_hbm_usage("Before loading model")
# def get_ref_maxtext_model():

#   #TODO: @mazumdera: change this to use Gemma2-2b-it
#   config = pyconfig.initialize(
#       ["", "../../maxtext/MaxText/configs/base.yml"], #TODO: @mazumdera: why decode.py?
#       base_output_directory="gs://dummy_output_dir",  # This is not used in Tunix.
#       run_name="none",
#       tokenizer_path="../../maxtext/assets/tokenizer.gemma",
#       per_device_batch_size=1,
#       max_target_length=1024,
#       steps=10,
#       async_checkpointing="false",
#       model_name="llama3.1-8b", #"llama3.1-8b"
#       checkpoint_period=5,
#       skip_jax_distributed_system="true",
#       weight_dtype="bfloat16",
#       attention="dot_product"
#   )

#   def create_model(config):
#     return mt.from_pretrained(config, rngs=nnx.Rngs(params=0, dropout=1))

#   model = nnx.eval_shape(create_model, config=config)

#   abstract_model = nnx.eval_shape(create_model, config=config)
#   graphdef, abstract_state = nnx.split(abstract_model)
#   print('The abstract NNX state (all leaves are abstract arrays):')
#   nnx.display(abstract_state)

#   @nnx.jit
#   def partial_init(config):
#     model = create_model(config)
#     # nnx.update(model, checkpoint)
#     # shard model
#     state = nnx.state(model)
#     specs = nnx.get_partition_spec(state)
#     state = jax.lax.with_sharding_constraint(state, specs)
#     nnx.update(model, state)
#     return model

#   with jax.sharding.use_mesh(model.mesh), nn.logical_axis_rules(config.logical_axis_rules):
#     model = partial_init(config)
#   print(model)

#   tunix_model = TunixMaxTextLlama(
#         base_model=model,
#         use_attention_mask=False,  # trust Tunix loss masking
#     )
#   mesh  = tunix_model.base.mesh

#   tunix_model.to_hf_mappings = lambda *args: {}
#   tunix_model.to_hf_transpose_keys = lambda *args: {}
#   tunix_model.lora_to_hf_mappings = lambda *args: {}

#   # Add these lines to properly get the graph definition and state
#   graphdef, state = nnx.split(tunix_model)
#   tunix_model = nnx.merge(graphdef, state)  # Recreate model in proper NNX format

#   return tunix_model, mesh

# model, mesh = get_ref_maxtext_model()

# print(model)
# show_hbm_usage("After loading model")
# nnx.display(nnx.state(model))